# 💼 Real-World Fintech Interview Cases (Practice)

Welcome to your **Fintech Business Interview Practice Notebook**!

In technical interviews for Data Analyst / Data Scientist roles, you will face business questions from stakeholders. Always follow the **Reverse Engineering Framework**:

```
┌──────────────────────────────┐     ┌──────────────────────────────┐     ┌──────────────────────────────┐
│ 1. Business Target & Schema  │ ──> │ 2. Audit Raw Data Quirks     │ ──> │ 3. Backward Code Blueprint   │
│ (Stakeholder Deliverable)    │     │ (Types, NaNs, Whitespace)    │     │ (Filter, Group, Aggregate)   │
└──────────────────────────────┘     └──────────────────────────────┘     └──────────────────────────────┘
```

---
### 📁 Available Datasets in `./data/`
- `data/customers.csv` (Customer demographics, KYC status, credit score, income, balances, account tier, PEP flag)
- `data/raw_transactions.csv` (Amounts, card types, statuses, device types, fraud flags, dates, regions)
- `data/merchants.csv` (Merchant categories, fee structures, risk levels, monthly volume)
- `data/disputes.csv` (Dispute records, chargebacks, fee tracking, resolution dates)

In [2]:
# Run this setup cell first
import pandas as pd
import numpy as np
import os

DATA_DIR = 'data'
print('✅ Environment ready! Available files:', os.listdir(DATA_DIR))

✅ Environment ready! Available files: ['customers.csv', 'disputes.csv', 'merchants.csv', 'raw_transactions.csv']


---
## 🏢 Case 1 (Compliance / AML Audit): High-Risk Customer Escalation

### 📌 Business Scenario
> **Stakeholder:** Head of Financial Crimes & AML Compliance  
> *"We are preparing for an audit. I need an escalation report of all unverified or high-exposure accounts. Specifically, find customers who are Politically Exposed Persons (`is_pep == 1`) OR have an account balance of at least $50,000 while their KYC is NOT verified."*


In [3]:
# ✍️ YOUR CODE HERE FOR CASE 1:
# 1. Load data/customers.csv
df = pd.read_csv('data/customers.csv')

# 2. Clean types, whitespace, and create full_name
df['country_code'] = df['country_code'].astype(str).str.strip()
df['kyc_status'] = df['kyc_status'].astype(str).str.strip()
df['full_name'] = df['first_name'].astype(str).str.strip() + ' ' + df['last_name'].astype(str).str.strip()
df['account_balance'] = pd.to_numeric(df['account_balance'], errors='coerce').fillna(0)
df['is_pep'] = pd.to_numeric(df['is_pep'], errors='coerce').fillna(0)

# 3. Apply the AML compliance filter mask
mask = (df['is_pep'] == 1) | ((df['account_balance'] >= 50000) & (df['kyc_status'] != 'Verified'))

# 4. Sort descending by account_balance and display top 10
df_audit = df[mask].sort_values(by='account_balance', ascending=False).head(10)[['customer_id', 'full_name', 'country_code', 'kyc_status', 'account_balance', 'is_pep']]
df_audit

,customer_id,full_name,country_code,kyc_status,account_balance,is_pep
123,C38746,Lisa Chen,DE,Rejected,74983.89,0
4371,C54241,Robert Clark,FR,Rejected,74953.93,0
3230,C28144,Carlos Walker,DE,Pending,74952.45,0
2183,C13282,Betty Chen,IN,Pending,74941.98,0
4785,C62928,James Kim,FR,Pending,74928.74,0
2103,C72162,William Tanaka,SG,Under Review,74873.83,0
4773,C21940,Joshua Lewis,US,Rejected,74790.50,0
1808,C68117,Sandra Davis,DE,Rejected,74782.88,0
1298,C98188,Michelle Anderson,SG,Rejected,74770.25,0
3412,C75602,Linda Davis,IN,Pending,74742.46,0


---
## 🏢 Case 2 (Merchant Risk Operations): High-Exposure Merchant Watchlist

### 📌 Business Scenario
> **Stakeholder:** Director of Merchant Risk  
> *"We need to place high-risk merchants under active review. Give me a shortlist of merchants in high-volatility industries ('Crypto & Digital Assets' or 'Gaming & Virtual Goods') that either have an 'Extreme' risk rating OR are marked for chargeback monitoring (`is_chargeback_monitored == 1`), and generate over $1,000,000 in monthly estimated volume."*


In [4]:
# ✍️ YOUR CODE HERE FOR CASE 2:
# 1. Load data/merchants.csv
df = pd.read_csv('data/merchants.csv')

# 2. Clean string columns (category, risk_rating)
df['category'] = df['category'].astype(str).str.strip()
df['risk_rating'] = df['risk_rating'].astype(str).str.strip()
df['is_chargeback_monitored'] = pd.to_numeric(df['is_chargeback_monitored'], errors='coerce')
df['monthly_volume_est'] = pd.to_numeric(df['monthly_volume_est'], errors='coerce').fillna(0)

# 3. Build compound boolean mask
mask = (df['category'].isin(['Crypto & Digital Assets', 'Gaming & Virtual Goods'])) & ((df['risk_rating'] == 'Extreme') | (df['is_chargeback_monitored'] == 1)) & (df['monthly_volume_est'] > 1000000)

# 4. Filter, sort by monthly_volume_est descending, and display top 5
df_risk = df[mask].sort_values(by='monthly_volume_est', ascending=False).head(5)[['merchant_id', 'merchant_name', 'category', 'risk_rating', 'monthly_volume_est']]
df_risk

,merchant_id,merchant_name,category,risk_rating,monthly_volume_est
1501,M6954,Nexus Solutions,Crypto & Digital Assets,Extreme,2487654.59
492,M5294,BlueSky Ventures,Gaming & Virtual Goods,High,2485170.34
1160,M7997,Beacon Mart,Crypto & Digital Assets,Extreme,2484261.65
853,M6737,Starlight Holdings,Crypto & Digital Assets,Extreme,2481367.18
279,M1854,Horizon Direct,Crypto & Digital Assets,Extreme,2480787.55


---
## 🏢 Case 3 (Product & Growth): Customer Tier Summary Scorecard

### 📌 Business Scenario
> **Stakeholder:** Product Growth Lead  
> *"We are designing premium card benefits. Can you summarize our customer base across each `account_tier`? For each tier, I need the total customer count, average annual income, and how many customers hold a crypto wallet."*


In [5]:
# ✍️ YOUR CODE HERE FOR CASE 3:
# 1. Load data/customers.csv
df = pd.read_csv('data/customers.csv')

# 2. Clean whitespace in account_tier and coerce annual_income to numeric
df['account_tier'] = df['account_tier'].astype(str).str.strip()
df['annual_income'] = pd.to_numeric(df['annual_income'], errors='coerce').fillna(0)
df['has_crypto_wallet'] = pd.to_numeric(df['has_crypto_wallet'], errors='coerce').fillna(0)

# 3. Group by account_tier and aggregate metrics
tier_summary = (df.groupby('account_tier').agg(
    total_customers=('customer_id', 'count'),
    avg_income=('annual_income', 'mean'),
    crypto_wallet_users=('has_crypto_wallet', 'sum')
).reset_index())

# 4. Round avg_income, sort by total_customers descending, and display
tier_summary['avg_income'] = tier_summary['avg_income'].round(2)
tier_summary = tier_summary.sort_values(by='total_customers', ascending=False)
tier_summary

,account_tier,total_customers,avg_income,crypto_wallet_users
4,VIP,1017,168675.55,308
3,Standard,1005,171224.44,273
1,Platinum,1000,171183.43,289
0,Gold,966,171255.94,281
2,Silver,929,170118.78,263


---
## 🏢 Case 4 (Payment Operations): Channel Performance & Ticket Sizes

### 📌 Business Scenario
> **Stakeholder:** Head of Payment Operations  
> *"Our POS terminals had issues yesterday. Please generate a breakdown of transactions across each `device_type` (`ATM`, `Desktop`, `Mobile`, `POS`). We need total transactions processed, total transaction volume in USD, and average ticket size."*


In [6]:
# ✍️ YOUR CODE HERE FOR CASE 4:
# 1. Load data/raw_transactions.csv
df = pd.read_csv('data/raw_transactions.csv')

# 2. Clean whitespace in device_type and coerce transaction_amount to numeric
df['transaction_amount'] = pd.to_numeric(df['transaction_amount'], errors='coerce').fillna(0)
df['device_type'] = df['device_type'].astype(str).str.strip()

# 3. Group by device_type and aggregate count, sum, and mean
grouped_summary = (df.groupby('device_type').agg(
    total_transactions=('transaction_id', 'count'),
    total_volume_usd=('transaction_amount', 'sum'),
    avg_ticket_size=('transaction_amount', 'mean')
).reset_index())

# 4. Round numerical values and sort descending by total_volume_usd
grouped_summary['total_volume_usd'] = grouped_summary['total_volume_usd'].round(2)
grouped_summary['avg_ticket_size'] = grouped_summary['avg_ticket_size'].round(2)
summary = grouped_summary.sort_values(by='total_volume_usd', ascending=False)
summary

,device_type,total_transactions,total_volume_usd,avg_ticket_size
3,POS,3815,3740466.26,980.46
2,Mobile,3781,3568186.65,943.72
0,ATM,3685,3516183.08,954.19
1,Desktop,3719,3502099.51,941.68


---
## 🏢 Case 5 (Cross-Table Merging & Risk Analysis): Merchant Dispute Exposure

### 📌 Business Scenario
> **Stakeholder:** VP of Merchant Risk  
> *"We need to identify which merchants generate the most chargebacks and disputed dollar volume. Join our dispute records (`disputes.csv`) with our merchant database (`merchants.csv`). For each merchant, calculate the total number of disputes, total disputed amount ($), and average dispute amount."*


In [7]:
# ✍️ YOUR CODE HERE FOR CASE 5:
# 1. Load data/disputes.csv and data/merchants.csv


# 2. Clean types / whitespace


# 3. Merge dataframes on merchant_id


# 4. Group by merchant and compute metrics


# 5. Sort descending by total_disputed_usd and show top 5


---
## 🏢 Case 6 (Time-Series & Date Arithmetic): Dispute Resolution SLA Performance

### 📌 Business Scenario
> **Stakeholder:** Head of Dispute & Chargeback Operations  
> *"Our compliance team requires us to resolve customer disputes within strict timelines. For all resolved disputes (where `resolution_date` is not missing), calculate the turnaround time in days (`resolution_date - dispute_date`). Summarize by `dispute_reason` to find average and maximum days taken to resolve."*


In [8]:
# ✍️ YOUR CODE HERE FOR CASE 6:
# 1. Load data/disputes.csv
df = pd.read_csv('data/disputes.csv')

# 2. Convert dispute_date and resolution_date to datetime
df['dispute_reason'] = df['dispute_reason'].astype(str).str.strip()
df['dispute_date'] = pd.to_datetime(df['dispute_date'])
df['resolution_date'] = pd.to_datetime(df['resolution_date'])

# 3. Filter for resolved cases and calculate turnaround_days
resolved_df = df[df['resolution_date'].notna()].copy()
resolved_df['turnaround_days'] = (resolved_df['resolution_date'] - resolved_df['dispute_date']).dt.days

# 4. Group by dispute_reason and compute SLA metrics
sla_summary = (resolved_df.groupby('dispute_reason').agg(
    resolved_count=('dispute_id', 'count'),
    avg_turnaround_days=('turnaround_days', 'mean'),
    max_turnaround_days=('turnaround_days', 'max')
).reset_index())

# 5. Sort descending by avg_turnaround_days
sla_summary['avg_turnaround_days'] = sla_summary['avg_turnaround_days'].round(1)
sla_summary = sla_summary.sort_values(by='avg_turnaround_days', ascending=False)
sla_summary


,dispute_reason,resolved_count,avg_turnaround_days,max_turnaround_days
0,Credit Not Processed,846,34.0,60
4,Item Not Received,925,33.6,60
6,Subscription Cancelled,902,33.5,60
3,Friendly Fraud / Unrecognized,916,33.2,60
2,Fraudulent Transaction,1234,32.9,60
5,Product Defective / Unacceptable,906,32.8,60
1,Duplicate Processing,901,32.5,60


---
## 🏢 Case 7 (Customer Segmentation & Binning): Credit Score Risk Profiling

### 📌 Business Scenario
> **Stakeholder:** Credit Risk Modeling Lead  
> *"We are launching an unsecured credit line. I need customer segmentation across standardized FICO credit score brackets (`Poor`, `Fair`, `Good`, `Very Good`, `Exceptional`). For each tier, calculate customer count, average credit score, average annual income, and total deposit balance."*


In [9]:
# ✍️ YOUR CODE HERE FOR CASE 7:
# 1. Load data/customers.csv
df = pd.read_csv('data/customers.csv')
df['credit_score'] = pd.to_numeric(df['credit_score'], errors='coerce')
df['annual_income'] = pd.to_numeric(df['annual_income'], errors='coerce')
df['account_balance'] = pd.to_numeric(df['account_balance'], errors='coerce')

# 2. Bin credit_score into tiers using pd.cut
bins = [300, 579, 669, 739, 799, 850]
labels = ['Poor', 'Fair', 'Good', 'Very Good', 'Exceptional']
df['credit_tier'] = pd.cut(df['credit_score'], bins=bins, labels=labels, include_lowest=True)

# 3. Group by credit_tier and compute aggregations
credit_summary = (df.groupby('credit_tier', observed=False).agg(
    customer_count=('customer_id', 'count'),
    avg_credit_score=('credit_score', 'mean'),
    avg_annual_income=('annual_income', 'mean'),
    total_balance_usd=('account_balance', 'sum')
).reset_index())

# 4. Format and display result
credit_summary['avg_credit_score'] = credit_summary['avg_credit_score'].round(1)
credit_summary['avg_annual_income'] = credit_summary['avg_annual_income'].round(2)
credit_summary['total_balance_usd'] = credit_summary['total_balance_usd'].round(2)
credit_summary


,credit_tier,customer_count,avg_credit_score,avg_annual_income,total_balance_usd
0,Poor,2119,464.8,170149.91,79993182.07
1,Fair,875,625.0,169462.55,33773596.79
2,Good,675,705.1,172642.61,25428605.78
3,Very Good,587,769.5,169634.76,22969342.37
4,Exceptional,526,824.8,168977.44,19827080.73


---
## 🏢 Case 8 (Pivot Tables & Multi-Dimensional Matrix): Regional Card Volume Heatmap

### 📌 Business Scenario
> **Stakeholder:** Head of Card Partnerships & Treasury  
> *"We are renegotiating interchange fees with Visa, MasterCard, and Amex. Create a cross-tabulated volume matrix showing total transaction dollar volume by `card_type` across each geographic `region` (`North`, `South`, `East`, `West`), including total sums across rows and columns."*


In [10]:
# ✍️ YOUR CODE HERE FOR CASE 8:
# 1. Load data/raw_transactions.csv
df = pd.read_csv('data/raw_transactions.csv')
df['card_type'] = df['card_type'].astype(str).str.strip()
df['region'] = df['region'].astype(str).str.strip().str.title()
df['transaction_amount'] = pd.to_numeric(df['transaction_amount'], errors='coerce').fillna(0)

# 2. Create pivot table with card_type as index, region as columns, and sum of transaction_amount
piv_df = df.pivot_table(
    index='card_type',
    columns='region',
    values='transaction_amount',
    aggfunc='sum',
    margins=True,
    margins_name='Total'
)

# 3. Include margins=True for total rows/columns and round to 2 decimals
piv_df = piv_df.round(2)
piv_df


region,East,North,South,West,Total
card_type,,,,,
Amex,912784.50,834262.34,909282.07,941216.55,3597545.46
Discover,920111.56,930155.72,850685.19,918999.53,3619952.00
MasterCard,935110.16,872961.89,882593.99,894422.01,3585088.05
Visa,880040.29,877655.68,909549.54,857104.48,3524349.99
Total,3648046.51,3515035.63,3552110.79,3611742.57,14326935.50


---
## 🏢 Case 9 (Window / Ranking Functions): Top 2 Highest Transactions Per Region

### 📌 Business Scenario
> **Stakeholder:** VIP Wealth Manager  
> *"We want to reward our biggest single spenders in every region. For each geographic `region`, identify the Top 2 single highest transaction records (completed transactions only)."*


In [11]:
# ✍️ YOUR CODE HERE FOR CASE 9:
# 1. Load data/raw_transactions.csv
df = pd.read_csv('data/raw_transactions.csv')
df['region'] = df['region'].astype(str).str.strip().str.title()
df['transaction_status'] = df['transaction_status'].astype(str).str.strip().str.title()
df['transaction_amount'] = pd.to_numeric(df['transaction_amount'], errors='coerce').fillna(0)

# 2. Filter for Completed transactions
df = df[df['transaction_status'] == 'Completed'].copy()

# 3. Compute dense rank per region on transaction_amount descending
df['region_rank'] = (
    df.groupby('region')['transaction_amount']
    .rank(method='dense', ascending=False)
    .astype(int)
)

# 4. Filter for rank <= 2 and sort by region, transaction_amount
top2_per_region = df[df['region_rank'] <= 2].sort_values(
    by=['region', 'transaction_amount'],
    ascending=[True, False]
)[[
    'region',
    'region_rank',
    'transaction_id',
    'customer_id',
    'transaction_amount',
    'card_type'
]]
top2_per_region


,region,region_rank,transaction_id,customer_id,transaction_amount,card_type
1422,East,1,TX106643,C89415,1998.49,MasterCard
1708,East,2,TX108086,C51467,1996.93,MasterCard
7498,East,2,TX104431,C27781,1996.93,MasterCard
3099,North,1,TX113444,C13478,1998.96,Visa
2221,North,2,TX105907,C28726,1996.02,Discover
11115,South,1,TX101707,C30635,1999.85,Amex
5848,South,2,TX113576,C78522,1997.32,MasterCard
12873,West,1,TX110389,C69429,1998.04,MasterCard
8800,West,2,TX110549,C56025,1995.46,MasterCard


---
## 🏢 Case 10 (String Manipulation & Regex Extraction): Email Domain Profiling

### 📌 Business Scenario
> **Stakeholder:** Fraud & Security Engineering Lead  
> *"We suspect fraudulent bot rings are registering using specific email domain patterns. Parse all customer emails to extract the email username and domain extension. Identify the top 5 most common email domains and calculate the percentage of users associated with each, while also flagging usernames containing numerical digits."*


In [ ]:
# ✍️ YOUR CODE HERE FOR CASE 10:
import pandas as pd

df=pd.read_csv('data/customers.csv')

df=df[df['email'].notna()].copy()
df['email']=df['email'].astype(str).str.strip().str.lower()
df['customer_id']=df['customer_id'].astype(str).str.strip()

# Split "user123@gmail.com" into username="user123" and domain="gmail.com"
df[['username','email_domain']]=df['email'].astype(str).str.split('@',expand=True)

# Boolean flag: True if contains digits, False otherwise
df['has_digits']=df['username'].str.contains(r'\d',regex=True)

#Group by Domain & Aggregate
domain_summary=df.groupby('email_domain').agg(
    total_users=('customer_id','count'),
    digit_username_count=('has_digits','sum')
).reset_index()

#Calculate Percentage & Take Top 5
total_customers=len(df)
domain_summary['pct_of_total_users']=(domain_summary['total_users']/total_customers*100).round(2)
top_5_domains=domain_summary.sort_values(by='total_users',ascending=False).head(5)
top_5_domains

# # 1. Quick sanity check on final result
# print(top_5_domains.info())

# # 2. Check value logic
# assert (top_5_domains['digit_username_count'] <= top_5_domains['total_users']).all(), "Error: digit count exceeds total users!"
# assert top_5_domains['pct_of_total_users'].sum() <= 100, "Error: percentages exceed 100%!"
# print("✅ All sanity checks passed!")


<class 'pandas.core.frame.DataFrame'>
Index: 5 entries, 2 to 4
Data columns (total 4 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   email_domain          5 non-null      object 
 1   total_users           5 non-null      int64  
 2   digit_username_count  5 non-null      int64  
 3   pct_of_total_users    5 non-null      float64
dtypes: float64(1), int64(2), object(1)
memory usage: 200.0+ bytes
None
✅ All sanity checks passed!


---
## 🏢 Case 11 (Data Cleaning & Missing Value Imputation): Dispute Settlement Audit

### 📌 Business Scenario
> **Stakeholder:** Payment Operations Lead  
> *"Our dispute records contain missing resolution dates and pending liability assignments. Perform a data completeness audit, calculate the missing rate for each field, impute missing liability assignments with 'Pending Investigation', and compute the total disputed amount and fee exposure by dispute status."*


In [13]:
# ✍️ YOUR CODE HERE FOR CASE 11:


---
## 🏢 Case 12 (Duplicate Detection & Anomaly Auditing): Double-Billing Detection

### 📌 Business Scenario
> **Stakeholder:** Customer Experience & Payment Protection Team  
> *"Customers are complaining about potential duplicate charges. Find all transactions where the exact same `customer_id`, `merchant_id`, and `transaction_amount` were billed on the exact same date (ignoring time if present, completed transactions only)."*


In [14]:
# ✍️ YOUR CODE HERE FOR CASE 12:


---
## 🏢 Case 13 (Conditional Logic & Vectorized Rules): Merchant Settlement Action Engine

### 📌 Business Scenario
> **Stakeholder:** Chief Risk Officer (CRO)  
> *"We need an automated settlement policy engine to classify merchants into payout risk actions: 'Immediate Hold', 'Daily Escrow Review', or 'Standard Auto-Payout' based on their chargeback monitoring status, risk rating, and monthly volume."*


In [15]:
# ✍️ YOUR CODE HERE FOR CASE 13:


---
## 🏢 Case 14 (Data Reshaping - Melting & Unstacking): Multi-Metric Regional Fraud Dashboard

### 📌 Business Scenario
> **Stakeholder:** BI & Executive Dashboard Team  
> *"Our executive reporting tool requires data in normalized unstacked and melted formats. For each `region` and `card_type`, calculate total transaction count, total fraud transaction count, and fraud rate (%). Then reshape the table using melt/unstack for reporting visualization."*


In [16]:
# ✍️ YOUR CODE HERE FOR CASE 14:


---
## 🏢 Case 15 (Relational Anti-Joins & Inactivity Analysis): Churned / Inactive Onboarded Customers

### 📌 Business Scenario
> **Stakeholder:** VP of Growth & Product Marketing  
> *"We have onboarded thousands of customers, but some never performed a single transaction. Find all customers who have zero recorded transactions in raw_transactions.csv. Analyze their total idle account balance by account_tier and risk_tier."*


In [17]:
# ✍️ YOUR CODE HERE FOR CASE 15:


In [ ]:
# ✍️ CASE 15 (ALTERNATIVE: WITHOUT METHOD CHAINING / PROCEDURAL STEP-BY-STEP):


---
## 🏢 Case 16 (Time-Series Resampling & Rolling Windows): Daily Volume & 7-Day Moving Average

### 📌 Business Scenario
> **Stakeholder:** Head of Treasury & Financial Planning  
> *"To detect liquidity bottlenecks and anomalous volume surges, build a daily time-series analysis showing daily completed transaction volume, 7-day rolling moving average volume, and day-over-day percentage growth."*


In [18]:
# ✍️ YOUR CODE HERE FOR CASE 16:


---
## 🏢 Case 17 (Multi-Table Merging & Loss Ratio): Customer Lifetime Value vs. Dispute Exposure

### 📌 Business Scenario
> **Stakeholder:** Credit Risk & Underwriting Committee  
> *"We need a 360-degree customer risk profile combining customer demographics, transaction spending, and dispute claims. For each customer with transaction activity, calculate their total transaction volume, total disputed amount, and dispute loss ratio (disputed amount / total transaction amount * 100). Identify the top 10 highest-dispute customers."*


In [19]:
# ✍️ YOUR CODE HERE FOR CASE 17:


---
## 🏢 Case 18 (Cross-Tabulation & Proportions): Dispute Reason vs. Liability Matrix

### 📌 Business Scenario
> **Stakeholder:** Merchant Chargeback Operations Lead  
> *"Generate a normalized cross-tabulation matrix showing the distribution of liability assignments (Merchant, Customer, Payment Processor, Pending) across each dispute reason. Show both raw counts and row-wise percentage distributions."*


In [20]:
# ✍️ YOUR CODE HERE FOR CASE 18:
